In [25]:
from collections import Counter, defaultdict, deque
from datetime import timedelta
import json

In [26]:
# 1. Count + top-k: top 3 cities by number of rides
def top_cities(rides, k=3):
    return Counter(r["city"] for r in rides).most_common(k)

In [27]:
rides = [
    {"city": "New York", "driver_id": "Alice",   "fare": 15.0},
    {"city": "Cairo",    "driver_id": "Bob",     "fare": 8.5},
    {"city": "New York", "driver_id": "Charlie", "fare": 22.0},
    {"city": "Dubai",    "driver_id": "Diana",   "fare": 30.0},
    {"city": "Cairo",    "driver_id": "Eve",     "fare": 12.0},
    {"city": "New York", "driver_id": "Alice",   "fare": 18.0},  # Alice has 2 rides
    {"city": "Dubai",    "driver_id": "Diana",   "fare": 25.0},  # Diana has 2 rides
    {"city": "Cairo",    "driver_id": "Bob",     "fare": 9.0},   # Bob has 2 rides
    {"city": "Lagos",    "driver_id": "Ivy",     "fare": 5.0},
    {"city": "Dubai",    "driver_id": "Diana",   "fare": 40.0},  # Diana has 3 rides
]


In [28]:
 # 2. Group + aggregate: average fare per driver
def avg_fare_per_driver(rides):
    totals = defaultdict(lambda: [0.0, 0])          # driver -> [sum, count]
    for r in rides:
        t = totals[r["driver_id"]]
        t[0] += r["fare"]
        t[1] += 1
    return {d: s / c for d, (s, c) in totals.items()}

In [29]:
# 3. Dedup, keep the latest version of each record
def latest_per_id(events):
    latest = {}
    for e in events:
        cur = latest.get(e["id"])
        if cur is None or e["updated_at"] > cur["updated_at"]:
            latest[e["id"]] = e
    return list(latest.values())

In [30]:
# 4. Parse JSON lines safely, count bad records instead of crashing
def parse_lines(lines):
    good, bad = [], 0
    for line in lines:
        try:
            rec = json.loads(line)
            if "ride_id" not in rec:
                raise ValueError("missing ride_id")
            good.append(rec)
        except (json.JSONDecodeError, ValueError):
            bad += 1                                 # in production: send to a dead-letter queue
    return good, bad

In [31]:
# 5. Merge overlapping intervals: total online time of a driver
def merge_intervals(intervals):
    merged = []
    for start, end in sorted(intervals):
        if merged and start <= merged[-1][1]:
            merged[-1][1] = max(merged[-1][1], end)
        else:
            merged.append([start, end])
    return merged

In [32]:
# 6. Sessionize: a new session starts after a 30-minute gap
def count_sessions(timestamps, gap=timedelta(minutes=30)):
    ts = sorted(timestamps)
    if not ts:
        return 0
    return 1 + sum(1 for prev, cur in zip(ts, ts[1:]) if cur - prev > gap)

In [33]:
# 7. Sliding window: max rides in any 60-minute window
def max_in_window(timestamps, window=timedelta(hours=1)):
    q, best = deque(), 0
    for t in sorted(timestamps):
        q.append(t)
        while t - q[0] >= window:
            q.popleft()
        best = max(best, len(q))
    return best

In [34]:
# 8. Process a huge file in batches with a generator (constant memory)
def read_in_batches(path, size=10_000):
    batch = []
    with open(path) as f:
        for line in f:
            batch.append(line)
            if len(batch) == size:
                yield batch
                batch = []
    if batch:
        yield batch

In [35]:
# 9. Flatten nested JSON: {"a": {"b": 1}} -> {"a.b": 1}
def flatten(d, parent="", sep="."):
    out = {}
    for k, v in d.items():
        key = f"{parent}{sep}{k}" if parent else k
        if isinstance(v, dict):
            out.update(flatten(v, key, sep))
        else:
            out[key] = v
    return out

In [36]:
from collections import Counter
counts = Counter(r["city"] for r in rides)   # Counter({'Cairo': 2, 'Alex': 1})
counts.most_common(1)                        # [('Cairo', 2)]

[('New York', 3)]

In [37]:
counts

Counter({'New York': 3, 'Cairo': 3, 'Dubai': 3, 'Lagos': 1})

In [38]:
from collections import defaultdict

revenue = defaultdict(float)          # a missing key starts at 0.0
for r in rides:
    revenue[r["city"]] += r["fare"]

In [39]:
revenue

defaultdict(float,
            {'New York': 55.0, 'Cairo': 29.5, 'Dubai': 95.0, 'Lagos': 5.0})

In [ ]:
if "Cairo" not in revenue:
    revenue["Cairo"] = 0
revenue["Cairo"] += 80

In [42]:
revenue

{'Cairo': 80}

In [43]:
events = [
    {"ride_id": 1, "city": "Cairo", "status": "requested", "fare": 0,  "updated_at": 1},
    {"ride_id": 1, "city": "Cairo", "status": "completed", "fare": 80, "updated_at": 3},
    {"ride_id": 2, "city": "Cairo", "status": "completed", "fare": 50, "updated_at": 2},
    {"ride_id": 3, "city": "Alex",  "status": "completed", "fare": 40, "updated_at": 2},
    {"ride_id": 3, "city": "Alex",  "status": "cancelled", "fare": 0,  "updated_at": 5},
    {"ride_id": 4, "city": "Alex",  "status": "completed", "fare": 60, "updated_at": 4},
]

# Expected output:
# {"Cairo": {"completed_rides": 2, "revenue": 130},
#  "Alex":  {"completed_rides": 1, "revenue": 60}}

In [44]:
latest = {}
for e in events:
    cur = latest.get(e["ride_id"])
    if cur is None or e["updated_at"] > cur["updated_at"]:
        latest[e["ride_id"]] = e

In [45]:
latest

{1: {'ride_id': 1,
  'city': 'Cairo',
  'status': 'completed',
  'fare': 80,
  'updated_at': 3},
 2: {'ride_id': 2,
  'city': 'Cairo',
  'status': 'completed',
  'fare': 50,
  'updated_at': 2},
 3: {'ride_id': 3,
  'city': 'Alex',
  'status': 'cancelled',
  'fare': 0,
  'updated_at': 5},
 4: {'ride_id': 4,
  'city': 'Alex',
  'status': 'completed',
  'fare': 60,
  'updated_at': 4}}